# Environment Check

This notebook records the local Python, operating system, hardware, and tool availability for ErgoStudy AI. It uses the Python standard library and handles optional tools gracefully.

In [ ]:
import ctypes
import importlib.util
import os
import platform
import shutil
import subprocess
import sys


def run_optional_command(command):
    """Return command output without failing when a tool is unavailable."""
    executable = shutil.which(command[0])
    if executable is None:
        return f"{command[0]} is not available."
    try:
        result = subprocess.run(
            [executable, *command[1:]],
            capture_output=True,
            text=True,
            timeout=15,
            check=False,
        )
        output = result.stdout.strip() or result.stderr.strip()
        return output or f"{command[0]} returned no output."
    except (OSError, subprocess.SubprocessError) as error:
        return f"{command[0]} check could not run: {error}"


## Python environment

In [ ]:
print("Python version:", sys.version.replace("\n", " "))
print("Python executable:", sys.executable)
print("Implementation:", platform.python_implementation())
print("Architecture:", platform.architecture()[0])


## Operating system

In [ ]:
print("System:", platform.system())
print("Release:", platform.release())
print("Version:", platform.version())
print("Machine:", platform.machine())
print("Platform:", platform.platform())


## CPU and memory

In [ ]:
def total_memory_gib():
    """Return physical memory in GiB using safe standard-library APIs."""
    try:
        if sys.platform == "win32":
            class MemoryStatus(ctypes.Structure):
                _fields_ = [
                    ("length", ctypes.c_ulong),
                    ("memory_load", ctypes.c_ulong),
                    ("total_physical", ctypes.c_ulonglong),
                    ("available_physical", ctypes.c_ulonglong),
                    ("total_page_file", ctypes.c_ulonglong),
                    ("available_page_file", ctypes.c_ulonglong),
                    ("total_virtual", ctypes.c_ulonglong),
                    ("available_virtual", ctypes.c_ulonglong),
                    ("available_extended_virtual", ctypes.c_ulonglong),
                ]

            status = MemoryStatus()
            status.length = ctypes.sizeof(MemoryStatus)
            if not ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status)):
                return None
            return status.total_physical / (1024 ** 3)

        page_size = os.sysconf("SC_PAGE_SIZE")
        page_count = os.sysconf("SC_PHYS_PAGES")
        return page_size * page_count / (1024 ** 3)
    except (AttributeError, OSError, ValueError):
        return None


processor = platform.processor() or os.environ.get("PROCESSOR_IDENTIFIER", "Unavailable")
memory_gib = total_memory_gib()
print("Processor:", processor)
print("Logical CPU count:", os.cpu_count() or "Unavailable")
print("Total physical memory:", f"{memory_gib:.2f} GiB" if memory_gib else "Unavailable")


## GPU and CUDA

In [ ]:
print("NVIDIA GPU check:")
print(run_optional_command([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
]))
print("\nCUDA toolkit check:")
print(run_optional_command(["nvcc", "--version"]))

if importlib.util.find_spec("torch") is None:
    print("\nPyTorch is not installed, so PyTorch CUDA availability was not tested.")
else:
    try:
        import torch

        print("\nPyTorch version:", torch.__version__)
        print("PyTorch CUDA available:", torch.cuda.is_available())
        print("PyTorch CUDA build:", torch.version.cuda)
        if torch.cuda.is_available():
            print("PyTorch GPU:", torch.cuda.get_device_name(0))
    except Exception as error:
        print("\nPyTorch CUDA check could not run:", error)


## Package and command availability

In [ ]:
packages = [
    "jupyter",
    "notebook",
    "jupyterlab",
    "ipykernel",
    "psutil",
    "torch",
    "fastapi",
    "chromadb",
    "sentence_transformers",
]
commands = ["git", "jupyter", "ollama", "nvidia-smi", "nvcc"]

print("Python packages:")
for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"  {package}: {'available' if available else 'not available'}")

print("\nCommands on PATH:")
for command in commands:
    location = shutil.which(command)
    print(f"  {command}: {location or 'not available'}")


## Findings at scaffold creation

The computer runs 64-bit Windows 11 on an Intel Core i5-12600HX with 24 GB of installed memory. It has an NVIDIA GeForce RTX 3050 Laptop GPU with 6 GB of dedicated VRAM, plus Intel integrated graphics. The NVIDIA driver reports CUDA compatibility, but the CUDA toolkit and PyTorch are not installed.

Python 3.11, 3.12, and 3.14 installations were detected. Python 3.12 is the recommended initial project interpreter because it is broadly compatible with the planned libraries. A Jupyter core installation and IPython kernel exist under Python 3.11, but no Jupyter Notebook or JupyterLab interface is currently available on PATH. Ollama is not installed. These missing tools are acceptable for the scaffold stage and should be reviewed before later development begins.